# 01 · Data exploration (Train/Val only)
Checks the data before any modelling. Uses the 6 Abu Dhabi Train/Val sites; the test and external sites appear on the map only as outlines and their images are never opened.

## Environment
Runs in the `mangrove` conda env (`2_Code\environment.yml`; setup in `RUN_LOCALLY.md`).

## Paths
`paths.py` defines every folder in one place; Jupyter is launched from the SDP folder, so no paths need editing.
* `3_Data/`: small inputs (sites, labels, image list, folds)
* `6_Work/`: everything the pipeline rebuilds (images, chips, datasets), about 4 GB
* `4_Results/`: tables and figures

In [ ]:
import os, subprocess, sys
from pathlib import Path

for _d in [Path.cwd(), *Path.cwd().parents]:          # works from SDP, SDP/2_Code or SDP/2_Code/notebooks
    if (_d / 'paths.py').exists():
        CODE = _d; break
    if (_d / '2_Code' / 'paths.py').exists():
        CODE = _d / '2_Code'; break
else:
    raise FileNotFoundError('2_Code/paths.py not found: start Jupyter inside the SDP folder')
sys.path.insert(0, str(CODE))
import paths as P

PROJECT_ROOT, LAYOUT = P.PROJECT_ROOT, 'sdp'
WORK, SITES, LABELS, SCENES, FOLDS, QA_CSV = P.WORK, P.SITES, P.LABELS, P.SCENES, P.FOLDS, P.QA_CSV
DATASETS, RESULTS = P.DATASETS, P.RESULTS
TRAINVAL = P.TRAIN_VAL            # the Test_Dataset folder is never opened in these notebooks

def run_step(*args):
    """Run 2_Code/run_all.py with these arguments and show its output here, live."""
    env = {**os.environ, 'PYTHONUTF8': '1', 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([sys.executable, str(CODE / 'run_all.py'), *map(str, args)], cwd=PROJECT_ROOT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError('run_all.py failed: see the messages above')

print(P.describe())

In [ ]:
import json, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 11})
pd.set_option('display.width', 160)

In [ ]:
from chipio import load_chip, get_label, chip_path

def stretch(a, hi):
    return np.clip(np.nan_to_num(a) / hi, 0, 1)

def rgb(chip):        # true colour: B04, B03, B02
    im = chip['image']; return np.dstack([stretch(im[2], .25), stretch(im[1], .25), stretch(im[0], .25)])

def false_colour(chip):   # B08, B04, B03: healthy vegetation looks red
    im = chip['image']; return np.dstack([stretch(im[3], .45), stretch(im[2], .25), stretch(im[1], .25)])

def label_img(lab):   # 0 = sand colour, 1 = green, 255 (no label) = grey
    out = np.zeros(lab.shape + (3,)); out[lab == 0] = (.93, .91, .84); out[lab == 1] = (0, .55, .24); out[lab == 255] = (.6, .6, .6)
    return out

In [ ]:
tv = pd.read_csv(TRAINVAL / 'catalog.csv')
tv20 = tv[tv.year == 2020]
folds = pd.read_csv(FOLDS)
print(len(tv), 'Train/Val chips;', len(tv20), 'from 2020 carry the 10 m training label')

## 1 · Sites
Each Abu Dhabi cross-validation site has its own fold colour. Grey outlines are the locked test sites, purple outlines the external sites in other emirates.

In [ ]:
import geopandas as gpd
sites = gpd.read_file(SITES).merge(folds, left_on='name', right_on='site', how='left')
fold_colors = {1: '#2E5597', 2: '#3D8B5A', 3: '#E8A33D', 4: '#C0504D', 5: '#4BACC6'}
def colour(r):
    if r.role == 'cv': return fold_colors[int(r.fold)]
    return '#888888' if r.role == 'test_fixed' else '#7B5EA7'
sites['c'] = sites.apply(colour, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={'width_ratios': [1.4, 1]})
for ax, (x0, x1, y0, y1, title) in zip(axes, [(53.1, 55.0, 24.0, 25.0, 'Abu Dhabi coast'),
                                             (51.5, 56.5, 23.6, 26.2, 'All 13 UAE sites')]):
    sites.boundary.plot(ax=ax, color=sites.c, linewidth=2)
    for _, r in sites.iterrows():
        c = r.geometry.centroid
        show = (x0 < c.x < x1 and y0 < c.y < y1) and (title == 'Abu Dhabi coast' or r.role == 'external_fixed')
        if show:
            tag = f"fold {int(r.fold)}" if r.role == 'cv' else ('test, locked' if r.role == 'test_fixed' else 'external')
            ax.annotate(f"{r['name']}\n({tag})", (c.x, r.geometry.bounds[3]), fontsize=7, ha='center', va='bottom')
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_title(title + ('' if title == 'Abu Dhabi coast' else ' (Abu Dhabi names on the left)')); ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
plt.tight_layout(); plt.show()

## 2 · Two label maps of the same place
GMW v4.0.19 (10 m, our training label) against GMW v4.1.12 (~30 m, the Seeing Orbit reference) at Jubail Island, 2020. The right panel shows where they agree. The two maps disagree substantially, so scores near 1.0 are not reachable.

In [ ]:
import rasterio
from rasterio.windows import from_bounds
site = 'jubail_island'
b = sites[sites.name == site].geometry.iloc[0].bounds
def read_label(fname):
    with rasterio.open(LABELS / fname) as r:
        return r.read(1, window=from_bounds(*b, transform=r.transform))
a10, a30 = read_label('gmw_v4019_2020_uae.tif') == 1, read_label('gmw_v4112_2020_uae.tif') == 1
iou = (a10 & a30).sum() / max((a10 | a30).sum(), 1)
agree = np.zeros(a10.shape + (3,)) + .95
agree[a10 & a30] = (0, .55, .24); agree[a10 & ~a30] = (.18, .33, .59); agree[~a10 & a30] = (.91, .64, .24)
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, img, t in zip(axes, [a10, a30, agree], ['GMW v4.0.19 (10 m)', 'GMW v4.1.12 (~30 m)', f'Agreement (IoU {iou:.2f})']):
    ax.imshow(img if img.ndim == 3 else img, cmap=None if img.ndim == 3 else 'Greens', interpolation='nearest'); ax.set_title(t); ax.axis('off')
from matplotlib.patches import Patch
axes[2].legend(handles=[Patch(color=(0, .55, .24), label='both'), Patch(color=(.18, .33, .59), label='only 10 m'),
                        Patch(color=(.91, .64, .24), label='only 30 m')], loc='lower left', fontsize=8)
plt.suptitle(f'{site}, 2020', y=1.03); plt.tight_layout(); plt.show()

## 3 · Example chips
The 2020 chip with the most mangrove for each Train/Val site: true colour, false colour (vegetation shows red), NDVI (−0.2 to 0.8) and the 10 m label (green = mangrove, grey = no label).

In [ ]:
ordered = ['eastern_mangroves', 'jubail_island', 'ras_ghurab', 'ghantoot', 'al_aryam_islands', 'al_nouf']
ordered = [s for s in ordered if s in set(tv20.site)]   # sites present in the dataset
fig, axes = plt.subplots(len(ordered), 4, squeeze=False, figsize=(12, 3 * len(ordered)))
for i, site in enumerate(tqdm(ordered, desc='Loading chips')):
    row = tv20[tv20.site == site].sort_values('mangrove_pct_v4019', ascending=False).iloc[0]
    chip = load_chip(chip_path(TRAINVAL, row))
    ndvi = chip['image'][7 - 1]
    panels = [rgb(chip), false_colour(chip), ndvi, label_img(chip['label_v4019'])]
    titles = ['true colour', 'false colour', 'NDVI', '10 m label']
    for j, (p, t) in enumerate(zip(panels, titles)):
        ax = axes[i, j]
        ax.imshow(p, cmap='RdYlGn' if j == 2 else None, vmin=-.2 if j == 2 else None, vmax=.8 if j == 2 else None)
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0: ax.set_title(t)
        if j == 0: ax.set_ylabel(f"{site}\n{row.date}", fontsize=9)
plt.tight_layout(); plt.show()

## 4 · Class balance
Mangrove is a small share of each chip, so models are judged by IoU and F1 rather than accuracy: predicting "no mangrove" everywhere would be about 90% accurate.

In [ ]:
per_site = tv20.groupby('site').agg(chips=('chip_id', 'size'), mangrove_pct=('mangrove_pct_v4019', 'mean')).reindex(ordered)
per_fold = tv20.groupby('fold').agg(chips=('chip_id', 'size'), mangrove_pct=('mangrove_pct_v4019', 'mean'))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(per_site.index, per_site.mangrove_pct, color='#3D8B5A')
for i, (c, p) in enumerate(zip(per_site.chips, per_site.mangrove_pct)):
    axes[0].text(i, p, f'{c} chips', ha='center', va='bottom', fontsize=8)
axes[0].set_ylabel('mean mangrove % per chip (10 m label)'); axes[0].set_title('Per site, 2020 chips'); axes[0].tick_params(axis='x', rotation=25)
axes[1].bar(per_fold.index.astype(int).astype(str), per_fold.chips, color='#2E5597')
axes[1].set_xlabel('fold'); axes[1].set_ylabel('chips (2020)'); axes[1].set_title('Chips per fold')
plt.tight_layout(); plt.show()
display(per_site.round(1))

## 5 · NDVI alone
NDVI sampled from 60 Train/Val chips in three groups: mangrove (label = 1), water (not mangrove, MNDWI > 0.2) and other (not mangrove, dry land: sand, city, other plants). Mangrove is well separated from water and sand but overlaps other vegetation on the high side. That is where NDVI fails and the SWIR bands help.

In [ ]:
rng = np.random.default_rng(0)
sample = tv20.sample(min(60, len(tv20)), random_state=0)
vals = {'mangrove': [], 'water': [], 'other': []}
for _, row in tqdm(sample.iterrows(), total=len(sample), desc='Sampling pixels'):
    chip = load_chip(chip_path(TRAINVAL, row))
    lab = chip['label_v4019']; ok = chip['valid'] & (lab != 255)
    ndvi, mndwi = chip['image'][6], chip['image'][7]
    vals['mangrove'].append(ndvi[ok & (lab == 1)])
    vals['water'].append(ndvi[ok & (lab == 0) & (mndwi > .2)])
    vals['other'].append(ndvi[ok & (lab == 0) & (mndwi <= .2)])
fig, ax = plt.subplots(figsize=(10, 4.5))
for (k, v), c in zip(vals.items(), ['#3D8B5A', '#2E5597', '#C9A66B']):
    v = np.concatenate(v); v = rng.choice(v, min(len(v), 300_000), replace=False)
    ax.hist(v, bins=120, range=(-.6, 1), density=True, alpha=.55, color=c, label=f'{k} ({len(v):,} px)')
ax.set_xlabel('NDVI'); ax.set_ylabel('density'); ax.set_title('NDVI by class, 2020 Train/Val chips'); ax.legend()
plt.tight_layout(); plt.show()

## 6 · Haze across dates
Brightness of the darkest open water in the red band for every Train/Val image; clear water is near 0. Values above 0.05 (dashed line) indicate haze or sun glint, which makes the same mangrove look different between dates. The baselines are therefore run with and without the hazy images.

In [ ]:
qa = pd.read_csv(QA_CSV)
qa = qa[qa.site.isin(ordered)].copy(); qa['date'] = pd.to_datetime(qa.date)
qa['target'] = qa.target.astype(str)
fig, ax = plt.subplots(figsize=(11, 4.5))
order = qa.sort_values('date').target.drop_duplicates().tolist()   # e.g. 2018_Mar ... 2025_Nov, in time order
x = {t: i for i, t in enumerate(order)}
for k, (site, c) in enumerate(zip(ordered, ['#2E5597', '#3D8B5A', '#E8A33D', '#C0504D', '#4BACC6', '#7B5EA7'])):
    d = qa[qa.site == site].copy(); d['x'] = d.target.map(x) + (k - 2.5) * 0.06
    ax.plot(d.sort_values('x').x, d.sort_values('x').water_b04_p5, 'o-', color=c, label=site, alpha=.85)
ax.set_xticks(range(len(order)), order)
ax.axhline(.05, ls='--', color='k', lw=1); ax.text(-.3, .053, 'haze cut-off 0.05', fontsize=8)
ax.set_ylabel('dark-water red reflectance (5th percentile)'); ax.set_title('Haze by date, Train/Val sites')
ax.legend(fontsize=8, ncol=3); plt.tight_layout(); plt.show()
print('Hazy images (above 0.05):', int((qa.water_b04_p5 > .05).sum()), 'of', len(qa))

**Takeaways:** mangrove is a small minority class; NDVI separates it from water and sand but not from other plants; the two reference maps disagree; haze shifts brightness between dates. Next: `02_baselines.ipynb`.